# [검증] 진재언 위험지수 5변수 재현과 Cronbach's α — 2026-10-01 (2026-10-04 집계구 정의 수정)

- 작성자: 이정연 (`ljy`)
- 목적: 충돌 #2(지수 산출 방법론) 검증 — 보고서 서술만으로 5변수를 재현해 Cronbach's α를 독립 계산
- 입력: `data/raw/sk_통신인구_202507~202512_통합.csv` (6개월, outer join 생성본)
- **집계구 정의(2026-10-04 수정)**: 재언 님이 공유한 EDA v4 노트북(`notebooks/261001_EDA_skt_isolation_jhj.ipynb`, Cell 8)에서 `BLOCK_CD[:13]` 코드절단이 공식 집계구 경계와 13.2%만 일치함을 직접 확인했다. 집계구 경계 파일(`data/raw/reference_data/{강남,춘천}_집계구경계/bnd_oa_*.shp`)을 받아 `src/features/block_panel.py`를 공간조인(point-in-polygon) 방식으로 다시 작성했다 — 이 노트북도 그 결과로 다시 돌렸다.
- 대상 시군구: 강남구(11230)·춘천시(32010)
- **원 코드 미확인 — 각 변수 정의는 `src/features/risk_axes_v1.py` docstring의 가정을 따름**
- 다루지 않는 범위: 최종 위험지수 가중합, 지역 내부 백분위 정규화, 5분위 구간화

In [1]:
from pathlib import Path

import pandas as pd

from src.features.block_panel import aggregate_to_block, load_oa_boundaries
from src.features.risk_axes_v1 import (
    align_risk_direction,
    cronbach_alpha_standardized,
    reduce_to_block_level,
)

MONTHS = ["202507", "202508", "202509", "202510", "202511", "202512"]

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"

boundaries = load_oa_boundaries(raw_dir / "reference_data")
print(f"집계구 경계 폴리곤 수: {len(boundaries):,}")

frames = [
    pd.read_csv(
        raw_dir / f"sk_통신인구_{month}_통합.csv",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
    )
    for month in MONTHS
]
combined = pd.concat(frames, ignore_index=True)
print(f"격자×월 전체 행: {len(combined):,}")

panel = aggregate_to_block(combined, boundaries)
print(f"집계구×월 패널 행: {len(panel):,}, 고유 집계구 수: {panel['집계구'].nunique():,}")

집계구 경계 폴리곤 수: 1,698


격자×월 전체 행: 721,318


집계구×월 패널 행: 10,170, 고유 집계구 수: 1,695


In [2]:
block_level = reduce_to_block_level(panel, months_order=MONTHS)
aligned = align_risk_direction(block_level)

print("집계구 단위 변수 요약:")
print(block_level.describe().round(3))

print("\n방향 정렬 후 상관행렬 (Pearson):")
print(aligned.corr().round(3))

alpha_pearson = cronbach_alpha_standardized(aligned, method="pearson")
alpha_spearman = cronbach_alpha_standardized(aligned, method="spearman")
print(f"\nCronbach's α (Pearson, 표준화): {alpha_pearson:.3f}")
print(f"Cronbach's α (Spearman, 표준화): {alpha_spearman:.3f}")
print(f"항목 수: {aligned.shape[1]}, 유효 집계구 수: {aligned.dropna().shape[0]:,}")

print("\n항목 하나를 뺐을 때의 α (어느 항목이 발목을 잡는지 확인):")
for col in aligned.columns:
    subset = aligned.drop(columns=[col])
    a_p = cronbach_alpha_standardized(subset, method="pearson")
    a_s = cronbach_alpha_standardized(subset, method="spearman")
    print(f"  {col} 제외 → Pearson α={a_p:.3f}, Spearman α={a_s:.3f}")

집계구 단위 변수 요약:
            평탄도      야간비중       주말비      고령비중        추세
count  1695.000  1695.000  1695.000  1695.000  1695.000
mean      0.588     0.055     2.519     0.227    -0.013
std       0.052     0.013     4.274     0.053     0.035
min       0.436     0.023     0.130     0.106    -0.219
25%       0.553     0.047     0.780     0.190    -0.022
50%       0.580     0.054     1.215     0.228    -0.007
75%       0.623     0.063     2.563     0.254     0.004
max       0.830     0.114    66.172     0.615     0.211

방향 정렬 후 상관행렬 (Pearson):
         평탄도_위험  야간비중_위험  주말비_위험  고령비중_위험  추세_위험
평탄도_위험    1.000    0.837   0.288    0.194 -0.029
야간비중_위험   0.837    1.000   0.372    0.129 -0.048
주말비_위험    0.288    0.372   1.000    0.071  0.077
고령비중_위험   0.194    0.129   0.071    1.000  0.203
추세_위험    -0.029   -0.048   0.077    0.203  1.000

Cronbach's α (Pearson, 표준화): 0.570
Cronbach's α (Spearman, 표준화): 0.659
항목 수: 5, 유효 집계구 수: 1,695

항목 하나를 뺐을 때의 α (어느 항목이 발목을 잡는지 확인):
  평탄도_위험 제외 → Pearson α=0.382

## 결론 (2026-10-04 집계구 정의 수정 반영)

- **집계구 정의를 코드절단(`BLOCK_CD[:13]`, 1,639개)에서 SGIS 공식 경계 공간조인(1,695개)으로 교체했다.** 재언 님 EDA v4가 이 코드절단이 13.2%만 맞는다는 걸 직접 확인해줬다 — 이번에 받은 집계구 경계 shapefile로 우리 쪽 `block_panel.py`도 같은 방식으로 다시 짰다.
- **Cronbach's α = 0.570(Pearson) / 0.659(Spearman)** — 2026-10-01 결과(0.589/0.670)와 거의 같다. **집계구 정의를 고쳐도 결론은 바뀌지 않는다**: 둘 다 0.7 기준 미달.
- 상관행렬도 이전과 거의 동일하다: **평탄도_위험 ↔ 야간비중_위험 = 0.837**(이전 0.838)로 여전히 사실상 중복 변수다. 추세_위험은 나머지와 거의 무관(-0.03~0.08)하지만 고령비중과는 약한 양의 상관(0.203)이다.
- **항목 하나를 뺐을 때 α가 어떻게 바뀌는지**(Pearson 기준):

  | 제외한 항목 | α |
  | --- | ---: |
  | 평탄도_위험 | 0.382 (↓) |
  | 야간비중_위험 | 0.382 (↓) |
  | 주말비_위험 | 0.522 (↓) |
  | 고령비중_위험 | 0.571 (≈) |
  | **추세_위험** | **0.648 (↑, 최대)** |
  | (아무것도 안 뺌) | 0.570 |

  이전과 같은 패턴이다 — 평탄도·야간비중 중 하나를 빼면 α가 오히려 떨어지고(0.38 안팎), 추세를 빼는 게 α를 가장 많이 올리지만(0.570→0.648, Spearman은 0.659→0.734로 0.7을 넘는다) 그래도 Pearson 기준으로는 0.7 미달이다.
- **주말비_위험**의 방향(원값 그대로 사용, 높을수록 위험)은 집계구 정의를 바꿔도 다른 4개 변수와 모두 양의 상관(0.07~0.37)을 유지한다 — `docs/plans/261002_weekend_ratio_external_validation_plan.md`에서 행정동 단위로 확인한 방향과 일관된다.
- **결론**: 집계구 결합 방법을 재언 님 EDA v4가 검증한 공간조인 방식으로 고쳐도, 2026-10-01에 내린 결론(α 기준 미달, 평탄도·야간비중 중복, 추세가 α를 가장 많이 끌어올리지만 그래도 기준 미달)은 그대로 유지된다. 이번 수정으로 "집계구 정의가 달라서 α가 다르게 나온 것 아니냐"는 의심은 해소됐다 — 재언 님이 실제로 보고한 α=0.185와 크게 다른 건 집계구 정의 차이가 아니라 **주말비 변수의 부호 컨벤션 차이**(재언은 `-WKND`로 뒤집어서 계산에 넣었다가 외부검증에서 기각, 우리는 원값 그대로 사용) 때문이라는 게 더 명확해졌다.
- **다음 확인 필요**: 재언 님 원 코드(공간조인 스크립트)와 집계구 코드 체계(TOT_OA_CD)를 직접 대조해 이 재현이 정확히 같은 전제를 쓰는지 확인하는 것은 여전히 남아 있다.